In [1]:
import pandas as pd
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict


tokenizer = AutoTokenizer.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture
)

token_length = 512 # adjust to maximal token length

# restricting legth (make room for cls token and paragraph seperators)
TOK_LEN = token_length - 10

In [2]:
# read  data
df = pd.read_parquet('/raid/deallab/SF_RAG_Data/ASQA/train.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who won the 2016 ncaa football national championship? 

qa_pairs :
[{'context': "The 13–1 Alabama Crimson Tide won the game, holding off the undefeated Clemson Tigers 45–40 in the fourth quarter. Accompanied by a talented receiving corps, Clemson's Heisman Finalist quarterback Deshaun Watson had a historic performance, setting the record for most total yards in national championship game history, with 478 yards (405 passing / 73 rushing) against the nation's third-ranked defense in Alabama, breaking the record previously set by Vince Young in the 2006 Rose Bowl. Following the game, the AP Poll also named Alabama as its top team of the season, giving Alabama their fourth title in seven seasons. Both Clemson and Alabama finished the season 14–1.", 'question': "Who won the 2016 season's ncaa football national championship?", 'short_answers': array(['Clemson Tigers', '2016 Clemson Tigers football team',
        '2016 Clemson Tigers football', 'the Tigers', 'Clemson',
 

In [14]:
#parse tables to text
def get_table(table):
    table_text = []
    for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
        tr_text = tr.get_text()
        tr_text = re.sub(r'\n+',';',tr_text).strip(';')
        if not tr_text: continue
        if table_text == []:
            tr_text = '\n#### Table: ' + tr_text
        table_text.append(tr_text)

    return '\n'.join(table_text)

# parse pars to text
def get_p(par):
    p_text = par.get_text()
    p_text = p_text.replace('\n','')
    return p_text

def get_h(heading):
    h = heading.find(['h1', 'h2', 'h3','h4', 'h5'])
    try:
        heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
    except:
        print(heading)
        raise
    h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
    return h_text

#pars unordered lists to text
def get_ul(ul):
    list_text = []
    for li in ul.find_all('li'):
        list_text.append('* ' + li.get_text())
    return '\n'.join(list_text)

# pars ordered list to text
def get_ol(ol):
    list_text = []
    for i, li in enumerate(ol.find_all('li')):
        if li.get_text():
            list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
    return '\n'.join(list_text)
        

# parse whole document
def parse_document(doc):
    content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
    document  = []
    for cont in content:
        #stop condition
        if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4', 'h5'], id=['See_also', 'References']):
            break
        
        #get headining
        if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
            document.append(get_h(cont))
        # get par
        elif cont.name == 'p':
            par = get_p(cont)
            if par:
                document.append(par)
        #get ul
        elif cont.name == 'ul':
            document.append(get_ul(cont))
        #get ol
        elif cont.name == 'ol':
            document.append(get_ol(cont))
        #get table
        elif cont.name == 'table':
            if cont.has_attr('class') and 'metadata' in cont['class']: continue
            document.append(get_table(cont))
        # explore div
        elif cont.name == 'div':
            document.append(parse_document(cont))

    return '\n'.join(document).strip('\n')



In [15]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])

#creating question evidence pairs for retrival training (not implemented)
qe_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'evidence_id'])

#fetched_documents = {}
for idx, row in df.iterrows():
    if idx == 100: break # change number of document to chunk/process
    sample_id = row['sample_id']
    evidences = row['wikipages']
    q1 = row['ambiguous_question']
    q2 = defaultdict(list)
    for q in row['qa_pairs']:
        question = q['question']
        wikipage = q['wikipage']
        if not question or not wikipage: continue
        q2[wikipage].append(question)
    for evidence in evidences:
        url = evidence['url']
        #if url in fetched_documents: continue
        #fetched_documents[url] = []
        page = requests.get(url)
        
        # Create a BeautifulSoup object
        soup = BeautifulSoup(page.text, 'html.parser')
        # get title
        title = soup.find(id='firstHeading').get_text()
        
        #extract content
        content = soup.find(class_='mw-content-ltr')
        parsed_doc = parse_document(content)
        
        # chunk document
        documents = [[]]
        
        for par in re.split(r'(?=\n#{1,4})', parsed_doc):
            tokenized_par = tokenizer.encode(par, add_special_tokens = False)
            length = len(tokenized_par)
            if len(documents[-1]) + length < TOK_LEN:
                documents[-1].extend(tokenized_par)
            elif length > TOK_LEN:
                begin = 0 
                while begin < length:
                    if begin + TOK_LEN >= length:
                        documents.append(tokenized_par[begin:])
                        break
                    documents.append(tokenized_par[begin:begin + TOK_LEN])
                    begin += TOK_LEN - int(TOK_LEN * 0.1)
            else:
                documents.append(tokenized_par)
            
        print(len(documents))
        for doc in documents:
            doc_text = tokenizer.decode(doc)
            id = uuid4()
            #fetched_documents[url].append(id)
            evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, q1, doc_text]
        
        if title in q2:
            for question in q2[title]:
                for doc in documents:
                    doc_text = tokenizer.decode(doc)
                    id = uuid4()
                    #fetched_documents[url].append(id)
                    evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, question, doc_text]
                
#print(fetched_documents)
evidence_df

158
16
14
11
6
35
53
42
30
40
11
51
129
8
13
7
30
73
129
36
4
6
21
38
1
3
15
29
34
20
27
3
14
6
1
16
55
8
10
12
12
11
15
34
31
55
44
32
9
43
14
36
9
9
54
6
37
4
25
7
12
6
45
5
4
1
7
2
54
21
2
10
4
14
31
8
21
14
1
29
29
16
9
30
14
11
38
12
8
3
16
32
13
46
71
33
28
37
1
11
2
3
4
8
2
42
70
22
2
53
41
11
20
7
232
39
244
189
116
1
23
28
18
67
6
7
1
3
32
12
25
35
19
15
1
39
41
27
5
9
8
14
104
16
5
9
3
5
28
44
52
17
21
3
28
250
3
16
42
4
24


,id,sample_id,title,url,question,text
0,2e2728bd-87df-4e01-bd5c-f4f5ba61ea60,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,Bunk'd is an American comedy television series...
1,045bf90b-2648-423e-8a98-4015d002fe72,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
2,2824c336-2369-4d51-9321-1e2c307257a2,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: No.overallNo. inseasonTitle [1][...
3,8de978e9-c2be-420a-bff3-f63a70ad61f8,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,She assigns the counselors-in-training to giv...
4,3b16bb4b-4cca-4901-919d-68090dd5755e,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,"as Gladys, Casey Campbell as Murphy;Absent: N..."
...,...,...,...,...,...,...
7160,46a5459f-9545-4e79-8c55-01202f6c4ebd,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,\n#### Harassment\n\nThe casting of Asian-Amer...
7161,8deccc00-9f71-4d4b-af9d-36050c3a9b18,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,\n#### Table: Award;Date of ceremony;Category;...
7162,d67f4acb-576d-49ef-9af7-85f648c8e217,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,John Boyega;Nominated\nBest Actress;Daisy Ridl...
7163,8a2fdb36-82ee-47f6-af27-c9c5f0d88fc7,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,Fiction/Horror Film;John Williams;Nominated\n...


In [16]:
evidence_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv', index=False)